In [2]:
%pip install -U "mne[full]"


   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.5 MB 329.4 kB/s eta 0:00:22
   -- ------------------------------------- 0.5/7.5 MB 329.4 kB/s eta 0:00:22
   -- ------------------------------------- 0.5/7.5 MB 329.4 kB/s eta 0:00:22
   ---- ----------------------------------- 0.8/7.5 MB 359.9 kB/s eta 0:00:19
   ---- ----------------------------------- 0.8/7.5 MB 359.9 kB/s e

In [16]:
import mne
import numpy as np
import pandas as pd
from scipy import stats
import os

CARPETA_DATOS = './sujetos_eeg/'
sujetos = [f"{i:03}" for i in range(1, 11)]
runs = [4, 8, 12]

def rms(epocas):
    # Calcula el RMS: primero por época/canal y luego promedia épocas
    if len(epocas) == 0:
        return None

    # data es tipo (n_epocas, n_canales, n_times)
    data = epocas.get_data(copy=False)
    rms_por_epoca = np.sqrt(np.mean(data**2, axis=2))

    # Promedio a través de todas las épocas, por eso es q es axis=0
    return np.mean(rms_por_epoca, axis=0)

In [ ]:
nombre_base = f"sub-{'001'}_task-motion_run-{'4'}_eeg"
archivo_set = os.path.join(CARPETA_DATOS, f"{nombre_base}.set")
if os.path.exists(archivo_set):
    raw = mne.io.read_raw_eeglab(archivo_set, preload=True, verbose=False)
print(raw.info)

<Info | 8 non-empty values
 bads: []
 ch_names: Fc5, Fc3, Fc1, Fcz, Fc2, Fc4, Fc6, C5, C3, C1, Cz, C2, C4, C6, ...
 chs: 64 EEG
 custom_ref_applied: False
 dig: 67 items (3 Cardinal, 64 EEG)
 highpass: 0.0 Hz
 lowpass: 80.0 Hz
 meas_date: unspecified
 nchan: 64
 projs: []
 sfreq: 160.0 Hz
>


Solo tiene un lowpass filter para 80Hz, o sea que necesita ser filtrada los datos.

In [32]:
datos_izq = []
datos_der = []
canales_nombres = None

for s in sujetos:
    epocas_sujeto = []

    for r in runs:
        nombre_base = f"sub-{s}_task-motion_run-{r}_eeg"
        archivo_set = os.path.join(CARPETA_DATOS, f"{nombre_base}.set")

        if os.path.exists(archivo_set):
            raw = mne.io.read_raw_eeglab(archivo_set, preload=True, verbose=False)

            # preprocesado
            raw.filter(l_freq=13, h_freq=30, verbose=False)
            raw.set_eeg_reference('average', projection=False, verbose=False)

            events, event_id = mne.events_from_annotations(raw, verbose=False)
            print(event_id)

            if canales_nombres is None:
                canales_nombres = raw.ch_names

            epochs = mne.Epochs(raw, events, event_id={'TASK2T1': 2, 'TASK2T2': 3},
                                tmin=0.5, tmax=3, baseline=None, preload=True, verbose=False)

            epocas_sujeto.append(epochs)

    # concatenae los 3 runs
    if epocas_sujeto:
        epochs_totales = mne.concatenate_epochs(epocas_sujeto, verbose=False)

        res_izq = rms(epochs_totales['TASK2T1'])
        res_der = rms(epochs_totales['TASK2T2'])

        if res_izq is not None:
            datos_izq.append(res_izq)
        if res_der is not None:
            datos_der.append(res_der)

df_izq = pd.DataFrame(datos_izq, columns=canales_nombres)
df_der = pd.DataFrame(datos_der, columns=canales_nombres)
print('DataFrame izquierda')
print(df_izq)
print('DataFrame derecha')
print(df_der)

{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22612\3056487470.py:32: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_totales = mne.concatenate_epochs(epocas_sujeto, verbose=False)


{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22612\3056487470.py:32: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_totales = mne.concatenate_epochs(epocas_sujeto, verbose=False)


{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}


...

C:\Users\ASUS\AppData\Local\Temp\ipykernel_22612\3056487470.py:32: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_totales = mne.concatenate_epochs(epocas_sujeto, verbose=False)


{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22612\3056487470.py:32: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_totales = mne.concatenate_epochs(epocas_sujeto, verbose=False)


{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
{np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}
DataFrame izquierda
        Fc5       Fc3       Fc1       Fcz       Fc2       Fc4       Fc6  \
0  0.000009  0.000009  0.000009  0.000009  0.000008  0.000008  0.000009   
1  0.000006  0.000006  0.000006  0.000007  0.000006  0.000005  0.000005   
2  0.000010  0.000008  0.000007  0.000007  0.000007  0.000009  0.000009   
3  0.000005  0.000004  0.000004  0.000004  0.000004  0.000004  0.000004   
4  0.000011  0.000004  0.000004  0.000004  0.000005  0.000006  0.000007   
5  0.000017  0.000007  0.000007  0.000006  0.000006  0.000006  0.000011   
6  0.000007  0.000007  0.000006  0.000005  0.000005  0.000005  0.000006   
7  0.000014  0.000005  0.000004  0.000004  0.000004  0.000004  0.000005   
8  0.000027  0.000015  0.000011  0.000011  0.000013  0.000019  0.000043   
9  0.000007  0.00000

C:\Users\ASUS\AppData\Local\Temp\ipykernel_22612\3056487470.py:32: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_totales = mne.concatenate_epochs(epocas_sujeto, verbose=False)


In [33]:
def analisis_estadistico(df_izq, df_der):
    resultados_finales = []

    for col in df_izq.columns:
        grupo_izq = df_izq[col]
        grupo_der = df_der[col]

        # 1. Prueba de Normalidad (Shapiro-Wilk)
        # Extrae el p-valuse que está en la posición 1 del resultado
        p_shapiro_izq = stats.shapiro(grupo_izq)[1]
        p_shapiro_der = stats.shapiro(grupo_der)[1]

        # Verificación de normalidad
        normal = (p_shapiro_izq > 0.05) and (p_shapiro_der > 0.05)

        # 2. Prueba de Homocedasticidad (Levene)
        p_levene = stats.levene(grupo_izq, grupo_der)[1]
        homocedastico = p_levene > 0.05

        # 3. Prueba de hipótesis (Se elige q test se puede hacer)
        if normal and homocedastico:
            stat, p_valor = stats.ttest_ind(grupo_izq, grupo_der)
            prueba = "T-Student"
        else:
            stat, p_valor = stats.mannwhitneyu(grupo_izq, grupo_der)
            prueba = "Mann-Whitney"

        resultados_finales.append({
            "Canal": col,
            "Prueba": prueba,
            "p-valor": p_valor
        })

    df_resultados = pd.DataFrame(resultados_finales)

    # Filtrar significativos según p < 0.05
    df_significativos = df_resultados[df_resultados["p-valor"] < 0.05]

    return df_resultados, df_significativos

df_resultados, df_significativos = analisis_estadistico(df_izq, df_der)

In [38]:
print('Resulados completos')
print(df_resultados)

Resulados completos
   Canal        Prueba   p-valor
0    Fc5     T-Student  0.983218
1    Fc3  Mann-Whitney  0.850107
2    Fc1     T-Student  0.939514
3    Fcz     T-Student  0.988225
4    Fc2     T-Student  0.932486
..   ...           ...       ...
59   Po8     T-Student  0.885535
60    O1     T-Student  0.930834
61    Oz     T-Student  0.980523
62    O2     T-Student  0.991225
63    Iz     T-Student  0.983949

[64 rows x 3 columns]


In [39]:
print('Resultados con p > 0.05')
print(df_significativos)

Resultados con p > 0.05
Empty DataFrame
Columns: [Canal, Prueba, p-valor]
Index: []
